## Data Processing

This notebook merges four datasets from three sources:
- National Science Foundation (NSF) Science & Engineering Indicators, Publication Output: Trends and International Comparisons
    - Total S&E publications across all fields (Table SPBS-2)
    - Total S&E psychology publications (Table SPBS-15)
- World Bank, DataBank World Development Indicators
    - Selected indicators: Expenditure on tertiary education (% of government expenditure on education), GDP per capita (current US\$), GDP per capita, PPP (constant 2021 international \$), GDP per capita, PPP (current international \$), GNI per capita, Atlas method (current US$), GNI per capita, PPP (constant 2021 international \$), GNI per capita, PPP (current international $), Government expenditure on education, total (% of GDP), Government expenditure on education, total (% of government expenditure), Individuals using the Internet (% of population), Population, total, Research and development expenditure (% of GDP), Researchers in R&D (per million people), School enrollment, tertiary (% gross), Scientific and technical journal articles, Urban population (% of total population)
- Taiwan Directorate General of Budget, Accounting and Statistics
    - Selected indicators: Population (annual, individuals), GDP ($US)

In [6]:
import pandas as pd
import numpy as np
import pycountry
import plotly.express as px
import matplotlib.pyplot as plt
from io import BytesIO
import plotly.io as pio
import plotly.graph_objects as go
from matplotlib import font_manager
from IPython.display import HTML
import scipy
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings

Country names are standardized using ISO-3 country codes as a common key for easy merging. Functions for bidirectional conversions between country names and ISO-3 codes below:

In [7]:
def name_to_iso3(name):
    """
    Get the ISO 3-letter code for a country name.
    """
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

    
def iso3_to_name(iso):
    """
    Get the country name from an ISO 3-letter code.
    """
    if iso == 'XKX':  # XKX (Kosovo) is not included in pycountry's ISO database
        return 'Kosovo'
    country = pycountry.countries.get(alpha_3=iso)
    if country is None:
        raise KeyError(f"unresolved ISO-3 code: {iso!r}")
    return country.name

### Dataset 1/4, 2/4: NSF S&E Publications

In [8]:
# NSF labels requiring manual ISO-3 standardization
nsf_iso_map = {
    'Bahamas, The': 'BHS',
    'Holy See (Vatican City)': 'VAT',
    'Kosovo': 'XKX',
    'Montenegroc': 'MNE',
    'Russia': 'RUS',
    'Serbiac': 'SRB',
    'Serbia and Montenegrod': 'SRB',  # Defunct union; add to modern Serbia's code
    'Turkey': 'TUR',
    'Congo, Democratic Republic of the': 'COD',
    'Congo, Republic of the': 'COG',
    'Côte d’Ivoire': 'CIV',
    'Gambia, The': 'GMB',
    'São Tomé and Príncipe': 'STP',
    'Brunei': 'BRN',
    'Burma': 'MMR',
    'Gaza Stripe': 'PSE',   # NSF's footnoted label for Gaza
    'West Banke': 'PSE',    # NSF's footnoted label for the West Bank
}

# Standardize country names for visualization labels
country_name_fixes = {
    'Bolivia, Plurinational State of': 'Bolivia',
    'Brunei Darussalam': 'Brunei',
    'Cabo Verde': 'Cape Verde',
    'Congo, The Democratic Republic of the': 'Democratic Republic of the Congo',
    'Czechia': 'Czech Republic',
    'Iran, Islamic Republic of': 'Iran',
    "Korea, Democratic People's Republic of": 'North Korea',
    'Korea, Republic of': 'South Korea',
    "Lao People's Democratic Republic": 'Laos',
    'Micronesia, Federated States of': 'Micronesia',
    'Moldova, Republic of': 'Moldova',
    'North Macedonia': 'Macedonia',
    'Russian Federation': 'Russia',
    'Syrian Arab Republic': 'Syria',
    'Taiwan, Province of China': 'Taiwan',
    'Tanzania, United Republic of': 'Tanzania',
    'Türkiye': 'Turkey',
    'Venezuela, Bolivarian Republic of': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'Holy See (Vatican City State)': 'Vatican City',
    'Palestine, State of': 'Palestine',
}

# Regional/world aggregate rows present in every NSF SPBS table
non_country_rows = [
    "World",
    "North America",
    "Central America and Caribbean",
    "South America",
    "Europe",
    "EU-27 and United Kingdoma",
    "EU-27b", "Other Europe",
    "Other Europe, 2020",
    "Middle East",
    "Africa",
    "Asia",
    "Australia and Oceania",
    "Unassigned",
]

def load_nsf_table(path, sheet_name, value_name, drop_cols=None):
    """
    Load one NSF SPBS table (country rows + year columns), drop regional
    aggregates, then standardize to ISO-3.
    """

    warnings.filterwarnings(
    "ignore", 
    category=UserWarning, 
    module="openpyxl.styles.stylesheet"
)
    
    raw = pd.read_excel(path, sheet_name=sheet_name, header=3)
    raw = raw.rename(columns={raw.columns[0]: "Country"})

    if drop_cols:
        raw = raw.drop(columns=drop_cols)

    raw = raw[~raw["Country"].isin(non_country_rows)]

    year_cols = [c for c in raw.columns if isinstance(c, (int, float))]

    df = raw.melt(
        id_vars="Country",
        value_vars=year_cols,
        var_name="Year",
        value_name=value_name
    )
    df = df.dropna(subset=[value_name])
    df["Year"] = df["Year"].astype(int)

    df["iso_alpha"] = df["Country"].apply(name_to_iso3)
    df["iso_alpha"] = df["iso_alpha"].fillna(df["Country"].map(nsf_iso_map))

    unmatched_iso = df[df["iso_alpha"].isna()]
    print(f"{sheet_name}: unmatched ISO codes = {len(unmatched_iso)}")
    if len(unmatched_iso):
        display(unmatched_iso[["Country", value_name]])

    # Sum NSF labels sharing an ISO code
    df = df.groupby(["iso_alpha", "Year"], as_index=False)[value_name].sum()

    df["Country"] = df["iso_alpha"].apply(iso3_to_name)
    df["Country"] = df["Country"].replace(country_name_fixes)

    unmatched_names = df[df["Country"].isna()]
    print(f"{sheet_name}: unmatched country names = {len(unmatched_names)}")
    if len(unmatched_names):
        display(unmatched_names[["iso_alpha", value_name]])

    return df

In [13]:
# All S&E publications
se_all_df = load_nsf_table(
    "../data/raw/nsf/SE_all_2003-2022.xlsx",
    sheet_name="Table SPBS-2",
    value_name="SE_articles_total",
    drop_cols=["Income level"],
)

# Psychology S&E publications
psych_df = load_nsf_table(
    "../data/raw/nsf/SE_psych_2003-2022.xlsx",
    sheet_name="Table SPBS-15",
    value_name="Publications",
)

total_pubs = psych_df["Publications"].sum()
recent_pubs = psych_df.loc[
    psych_df["Year"].between(2020, 2022),
    "Publications"
].sum()

print(f"\nNSF psychology S&E countries retained: {psych_df['Country'].nunique()}")
print(f"NSF total S&E countries retained: {se_all_df['Country'].nunique()}")
print(f"NSF psychology S&E years included: {psych_df['Year'].min()}-{psych_df['Year'].max()}")
print(f"NSF total S&E countries years included: {se_all_df['Year'].min()}-{se_all_df['Year'].max()}")

print(f"\nTotal psychology publications, 2003-2022: {total_pubs:,.0f}")
print(f"Psychology publications, 2020-2022 subset: {recent_pubs:,.0f} ({recent_pubs / total_pubs:.1%} of all psychology publications)")

Table SPBS-2: unmatched ISO codes = 0
Table SPBS-2: unmatched country names = 0
Table SPBS-15: unmatched ISO codes = 0
Table SPBS-15: unmatched country names = 0

NSF psychology S&E countries retained: 201
NSF total S&E countries retained: 201
NSF psychology S&E years included: 2003-2022
NSF total S&E countries years included: 2003-2022

Total psychology publications, 2003-2022: 815,406
Psychology publications, 2020-2022 subset: 187,411 (23.0% of all psychology publications)


### Dataset 3/4: World Bank WDI

In [ ]:
wb_data = pd.read_csv('../data/raw/wbi/35cba9f8-915e-41cf-95b9-1ccc17410135_Data.csv')

wb_data.info()

In [ ]:
wb_year_cols = [
    col for col in wb_data.columns
    if '[YR' in col
]

for col in wb_year_cols:
    wb_data[col] = (
        wb_data[col]
        .replace('..', np.nan)
        .astype(float)
    )

wb_df = wb_data.melt(
    id_vars=[
        'Country Name',
        'Country Code',
        'Series Name'
    ],
    value_vars=wb_year_cols,
    var_name='Year',
    value_name='Value'
)

wb_df['Year'] = (
    wb_df['Year']
    .str.extract(r'(\d{4})')
    .astype(int)
)

wb_df['Value'] = pd.to_numeric(
    wb_df['Value'],
    errors='coerce'
)

wb_df = (
    wb_df
    .pivot_table(
        index=[
            'Country Name',
            'Country Code',
            'Year'
        ],
        columns='Series Name',
        values='Value',
        aggfunc='first'
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

wb_df.info()